# 🛡️ SOC Duel: Multi-Agent Reinforcement Learning in a Datacenter Environment

This notebook trains a dual-role AI agent (Defender vs. Adversary) using **GRPO (Group Relative Policy Optimization)** on top of a quantized `Gemma-4` model via `unsloth`. The agent learns to either mitigate threats as a SOC defender or escalate them as a chaos-monkey adversary, guided by a custom reward function with XAI (explainability) requirements.

---

## ⚙️ Section 1: Environment Setup
### Install Dependencies

Install all required packages: `unsloth` for efficient fine-tuning, `trl` for GRPO training, `openenv-core` for the simulation environment, and supporting libraries for async HTTP, plotting, and API serving.

In [ ]:
!pip install -q tqdm openenv-core unsloth trl matplotlib nest_asyncio fastmcp httpx fastapi uvicorn openai

### 🧹 Clean Up Previous Run Artifacts

Removes stale checkpoints, log files, and corrupted weights from any prior training runs to ensure a clean slate. This avoids conflicts during dataset loading and model saving.

In [ ]:
!rm -rf /kaggle/working/soc_lora
!rm -rf /kaggle/working/soc_grpo
!rm -f /kaggle/working/*.log
!rm -rf /kaggle/working/wandb

print("🧹 All SOC checkpoints, logs, and corrupted weights have been vaporized!")

---

## 📁 Section 2: Dataset & Source Code Loading
### Mount & Import the Datacenter Environment

Searches for the `datacenter_env.py` source file inside `/kaggle/input`, copies the full `Datacenter` package to the working directory, and imports it. This environment simulates a live datacenter topology that the agent will interact with during training.

In [ ]:
import os, sys, shutil, pathlib

WORK_PATH = pathlib.Path('/kaggle/working')
dst = WORK_PATH / 'Datacenter'

print("Searching for dataset in /kaggle/input...")
input_dir = pathlib.Path('/kaggle/input')
# Update this to match your actual file name
found_paths = list(input_dir.rglob('server/datacenter_env.py')) 

if not found_paths:
    raise RuntimeError('Could not find server/datacenter_env.py. Did you attach the dataset?')

src = found_paths[0].parent.parent
print(f"Found source code at: {src}")

shutil.copytree(
    src, 
    dst, 
    dirs_exist_ok=True, 
    ignore=shutil.ignore_patterns('.git', '__pycache__', '*.pyc')
)

print(f'Copied source code to {dst}')

os.chdir(WORK_PATH)
if str(WORK_PATH) not in sys.path:
    sys.path.insert(0, str(WORK_PATH))

# Import your package (change 'Datacenter' if your root folder is named differently)
import Datacenter 
print('Datacenter imported OK from', Datacenter.__file__)

---

## 🌐 Section 3: Simulation Server
### Launch the FastAPI Environment Server

Spins up the `uvicorn`-powered datacenter simulation server in a background subprocess on port `8000`. The server exposes REST endpoints that the reward function calls via `httpx` to simulate tool invocations (e.g., `scan_topology`, `migrate_workload`). A health-check loop polls until the server is ready.

In [ ]:
import nest_asyncio, subprocess, time, httpx, os, sys
nest_asyncio.apply()
ENV_URL = 'http://127.0.0.1:8000'

server_log = open('server_debug.log', 'w')

# Update the module path to your actual app file
SERVER_PROC = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'Datacenter.server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=server_log, 
    stderr=subprocess.STDOUT, 
    cwd=WORK_PATH, 
    env={**os.environ, 'PYTHONUNBUFFERED': '1'}
)

for i in range(30):
    try:
        if httpx.get(f'{ENV_URL}/health').status_code == 200: 
            break
    except: pass
    time.sleep(1)
    
print("✅ SOC Environment Server Healthy (Logs routed to server_debug.log)")

> **Note:** The server logs are routed to `server_debug.log`. Inspect that file if you encounter unexpected reward function errors.

---

## 🤖 Section 4: Model Loading & LoRA Adapter Setup
### Load Base Model with 4-bit Quantization + Attach LoRA Adapters

Loads `unsloth/gemma-4-E4B-it` in **4-bit NF4 quantization** to fit within Kaggle GPU memory limits. Then attaches lightweight **LoRA adapters** (rank 16) to the attention and MLP projection layers. Only these adapters are trained — the base model weights remain frozen.

| Parameter | Value |
|-----------|-------|
| Model | `gemma-4-E4B-it-unsloth-bnb-4bit` |
| Max Seq Length | 2048 |
| LoRA Rank (`r`) | 16 |
| LoRA Alpha | 16 |
| Quantization | 4-bit (BnB) |
| Gradient Checkpointing | ✅ Enabled (unsloth) |

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it-unsloth-bnb-4bit", 
    max_seq_length=2048,
    load_in_4bit=True,
    max_lora_rank=16,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma", 
)

model = FastLanguageModel.get_peft_model(
    model = model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

print("✅ Model, tokenizer, and adapters successfully loaded!")

---

## 🧠 Section 5: Reward Logic & Dual-Role Policy
### Core Reward Function, XAI Auditor & JSON Parser

This is the heart of the training loop. It defines:

- **`DEFENDER_PROMPT` / `ADVERSARY_PROMPT`**: System instructions for each role. Both require structured JSON output with mandatory reasoning fields (`threat_analysis`, `justification`, `thought`).
- **`verify_reasoning_integrity()`**: Checks whether the model's reasoning mentions node types that are consistent with the actual action taken. Hallucinated node references are penalized.
- **`_extract_soc_json()`**: A robust parser that strips model artifacts (e.g., `<end_of_turn>`, markdown fences) and extracts a valid `{tool, arguments}` JSON object from raw completions.
- **`soc_reward_fn()`**: The GRPO reward function. For each completion it:
  1. Parses the model's JSON output.
  2. Executes the tool call against the live simulation environment.
  3. Computes a composite reward across three buckets:

| Reward Bucket | Max Value | Description |
|---------------|-----------|-------------|
| Outcome | 0.50 | Threat score driven toward goal (↓ for defender, ↑ for adversary) |
| Integrity | 0.25 | XAI reasoning quality — penalized if empty or hallucinated |
| Stealth | 0.24 | Efficiency — penalized for noisy scans or shadow node blunders |

In [ ]:
# ==============================================================================
# CELL 7: Reward Logic, Semantic Auditor & Dual-Role Policy (ROBUST VERSION)
# ==============================================================================
import json
import re
import ast
import torch
from unsloth import FastLanguageModel
from openenv.core.env_server.mcp_types import CallToolAction
from tqdm.auto import tqdm

import sys
import os

# PATH INJECTION: Add the Datacenter root to sys.path so internal imports work
# This allows 'from server.soc_sim' to work inside 'datacenter_env.py'
datacenter_path = os.path.abspath("Datacenter")
if datacenter_path not in sys.path:
    sys.path.insert(0, datacenter_path)

# Now the import will work without triggering ModuleNotFoundError
from server.datacenter_env import DatacenterEnvironment

DEFENDER_PROMPT = """[ROLE: DEFENDER]
You are the SOC AI. Intruders are in the datacenter. Lower the Threat Score below 0.05.
Respond ONLY with valid JSON. Do NOT output conversational text.

MANDATORY XAI SCHEMA:
1. "threat_analysis" (string): Evaluate the noise and shadow nodes.
2. "justification" (string): Why are you taking this action?
(Missing reasoning incurs a -0.20 penalty. Hallucinating node tags incurs -0.05).

TOOLS: `scan_topology`, `enumerate_authorized_migrations`, `migrate_workload`
"""

ADVERSARY_PROMPT = """[ROLE: ADVERSARY]
You are the Chaos Monkey APT. Raise the Threat Score to 0.85.
Respond ONLY with valid JSON. Do NOT output conversational text.

MANDATORY XAI SCHEMA:
1. "thought" (string): Why are you targeting this specific node?
(Missing reasoning incurs a -0.10 stealth penalty. Hallucinating incurs -0.05).

TOOLS: `scan_topology`, `enumerate_authorized_migrations`, `migrate_workload`
"""

EPISODE_LOG = []

NODE_SYNONYMS = {
    "database": ["db", "sql", "datastore", "storage", "postgres", "redis", "cluster"],
    "web_server": ["frontend", "ui", "nginx", "apache", "webapp", "portal", "gateway"],
    "app_server": ["backend", "logic", "processing", "middleware", "compute"],
    "backup": ["recovery", "archive", "snapshot", "replica", "dr"],
    "shadow_node": ["decoy", "honeypot", "trap", "mirror", "fake"],
    "workload": ["task", "process", "container", "pod", "service", "target", "node"]
}

def verify_reasoning_integrity(thought_string, actual_tags):
    thought_lower = thought_string.lower()
    for tag, synonyms in NODE_SYNONYMS.items():
        if any(syn in thought_lower for syn in synonyms):
            if tag not in actual_tags and "workload" not in actual_tags:
                return False 
    return True

def _extract_soc_json(raw_text: str):
    if not isinstance(raw_text, str): return None
    text = raw_text.replace("<end_of_turn>", "").replace("<eos>", "").strip()
    
    if text.startswith("`" * 3): 
        text = re.sub(r"^```[a-zA-Z]*\\s*", "", text).rstrip("`").strip()

    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1: return None
    
    try:
        candidate = text[start:end+1]
        candidate = re.sub(r',\\s*}', '}', candidate)
        candidate = re.sub(r',\\s*\\]', ']', candidate)
        
        parsed_obj = json.loads(candidate)
        tool_val = parsed_obj.get("tool") or parsed_obj.get("action")
        args = parsed_obj.get("arguments", {})
        
        for key in ["threat_analysis", "justification", "thought", "target_id", "candidate_migrations"]:
            if key in parsed_obj and key not in args:
                args[key] = parsed_obj[key]
                
        return {"tool": tool_val, "arguments": args}
    except Exception:
        return None

def soc_reward_fn(prompts, completions, **kw):
    rewards = []
    env = DatacenterEnvironment()
    FastLanguageModel.for_inference(model)

    # Wrap loop in tqdm for progress tracking
    pbar = tqdm(enumerate(completions), total=len(completions), desc="Evaluating Rewards")
    for i, completion in pbar:
        try:
            raw_text = completion[0]['content'] if isinstance(completion, list) else completion
            parsed = _extract_soc_json(raw_text)
            
            prompt_text = prompts[i][-1]['content'] if isinstance(prompts[i], list) else str(prompts[i])
            is_defender = "[ROLE: DEFENDER]" in prompt_text
            role = "DEFENDER" if is_defender else "ADVERSARY"

            if parsed is None:
                print(f"   [{role}] ❌ Failed to parse JSON from model output.")
                rewards.append(0.01)
                continue

            tool_name = parsed.get("tool", "None")
            args = parsed.get("arguments", {})

            # 1. MOCK ENV EXECUTION
            env.reset()
            action = CallToolAction(tool_name=tool_name, arguments=args)
            
            print(f"   [{role}] 🔧 Tool: {tool_name}({json.dumps(args)})")
            
            obs = env.step(action)
            
            current_threat = getattr(env, 'adversary_threat', 0.40)
            scans_used = getattr(env, 'scans_this_turn', 0)
            actual_tags = getattr(env, 'last_target_tags', ["workload"])

            # 2. BUCKET CALCULATION
            outcome_reward = (1.0 - current_threat) * 0.50 if is_defender else current_threat * 0.50
            
            integrity_reward = 0.25
            thought_string = args.get("threat_analysis", "") + args.get("justification", "") if is_defender else args.get("thought", "")
            
            if len(thought_string.strip()) < 10:
                integrity_reward -= 0.20 if is_defender else 0.10
            elif not verify_reasoning_integrity(thought_string, actual_tags):
                integrity_reward -= 0.05
                
            stealth_reward = 0.24
            if is_defender:
                stealth_reward = max(0.0, 0.24 - (scans_used * 0.04)) 
            else:
                noise_increase = 0.0
                if tool_name == 'scan_topology': noise_increase = 1.0
                elif tool_name == 'migrate_workload' and 'shadow_node' in actual_tags: noise_increase = 2.0
                stealth_reward = max(0.0, 0.24 - (noise_increase * 0.05))

            total_reward = max(0.01, min(0.99, outcome_reward + integrity_reward + stealth_reward))
            
            print(f"   [{role}] 💰 Reward: {total_reward:.2f} (Outcome: {outcome_reward:.2f}, Integrity: {integrity_reward:.2f}, Stealth: {stealth_reward:.2f})")
            print(f"   [{role}] ✅ Action executed successfully\n")
            
            rewards.append(total_reward)
            EPISODE_LOG.append({'total': total_reward, 'stealth': stealth_reward, 'integrity': integrity_reward})

        except Exception as e:
            # If any specific completion fails (e.g. env.step error), catch it here
            # and append a floor reward so the trainer doesn't crash.
            print(f"   [{role}] ❌ Error during execution: {e}")
            rewards.append(0.01)

    env.close()
    FastLanguageModel.for_training(model)
    return rewards

print("✅ Robust SOC Reward function initialised.")

---

## 📊 Section 6: Dataset Construction
### Build the Dual-Role GRPO Training Dataset

Generates **4,000 training prompts** alternating 50/50 between Defender and Adversary roles. Each prompt is a single user-turn message combining the role system prompt with a randomized scenario (threat level, noise percentage, objective). The dataset is assembled into a HuggingFace `Dataset` object compatible with `GRPOTrainer`.

> **Design rationale:** Interleaving roles in a single dataset forces the model to learn context-sensitive behavior — acting defensively or offensively based solely on the `[ROLE: ...]` tag in the prompt.

In [ ]:
# ==============================================================================
# CELL 8: Construct the Dual-Role GRPO Dataset
# ==============================================================================
from datasets import Dataset
import random

TRAIN_ROWS = 4000
prompts = []

for i in range(TRAIN_ROWS):
    # Flip a coin: 50% chance to be Defender, 50% chance to be Adversary
    if i % 2 == 0:
        system_msg = DEFENDER_PROMPT
        scenario = f"WARNING: Intruders detected. Threat Level: 0.40. Noise Level: {random.randint(30, 80)}%. Begin mitigation."
    else:
        system_msg = ADVERSARY_PROMPT
        scenario = f"OBJECTIVE: Escalate privileges. Threat Level is 0.40. Noise fields active. Begin lateral movement."

    prompts.append({
        'prompt': [
            {
                'role': 'user', 
                'content': f"{system_msg}\n\n═══════════════════════════════════════════════════\n{scenario}"
            }
        ]
    })

train_ds = Dataset.from_list(prompts)
print(f"✅ Dual-Role SOC dataset built with {len(train_ds)} rows (50/50 Split).")

---

## 🚀 Section 7: GRPO Training
### Configure & Launch the GRPOTrainer

Sets up `GRPOConfig` and kicks off training. Key hyperparameters:

| Hyperparameter | Value | Rationale |
|----------------|-------|-----------|
| `per_device_train_batch_size` | 1 | Memory-constrained; offset by gradient accumulation |
| `gradient_accumulation_steps` | 4 | Effective batch size = 4 |
| `num_generations` | 2 | Completions sampled per prompt for relative scoring |
| `max_prompt_length` | 1024 | Covers full role prompt + scenario |
| `max_completion_length` | 512 | Sufficient for JSON + reasoning fields |
| `learning_rate` | 5e-6 | Conservative to preserve base model knowledge |
| `num_train_epochs` | 10 | Hard cap via `max_steps=50` |
| `bf16` | Auto-detected | Falls back to `fp16` on older GPUs |

In [ ]:
# ==============================================================================
# CELL 9: GRPOTrainer Setup (Modified for 10 Epochs / 50 Steps)
# ==============================================================================
import torch
from trl import GRPOTrainer, GRPOConfig

OUTPUT_DIR = '/kaggle/working/soc_grpo'
USE_BF16 = torch.cuda.is_bf16_supported()

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=2,
    max_prompt_length=1024,
    max_completion_length=512, 
    learning_rate=5e-6,
    logging_steps=1,
    # --- Modified constraints ---
    num_train_epochs=10, 
    max_steps=50, 
    # ----------------------------
    save_steps=10,
    save_total_limit=1, 
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    report_to='none',
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[soc_reward_fn], 
    args=grpo_config,
    train_dataset=train_ds,
)

print("🚀 Starting SOC GRPO Training (10 Epochs / 50 Max Steps)...")
trainer.train()

> ⏱️ Training may take several minutes per epoch on a Kaggle T4/P100. Monitor the live loss logs printed to stdout.

---

## 📈 Section 8: Log Analysis & Reward Visualization
### Parse Training Logs & Plot Multi-Panel Performance Dashboard

Reads `Logs.txt` produced during training and extracts:
- **Per-step reward breakdown** (Outcome, Integrity, Stealth components)
- **Hallucination events** detected by the Layer 6 Semantic Audit

Generates a two-panel dark-themed figure:
1. **Top panel**: Smoothed total reward + individual component curves over inference time.
2. **Bottom panel**: Hallucination frequency histogram + cumulative audit failure count.

The summary stats (total samples, max/avg reward, total audit failures) are overlaid as a text box.

In [ ]:
import re
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime

def parse_log_file(file_path):
    """
    Parses the log file to extract rewards and hallucination events.
    """
    data = []
    hallucinations = []
    
    # Regex patterns
    # Matches: 42672.3s 1124 [REWARD] Outcome:0.350 | Integrity:0.250 | Stealth:0.200
    reward_pattern = re.compile(
        r"(\d+\.?\d*)s\s+\d+\s+\[REWARD\]\s+Outcome:(\d+\.\d+)\s+\|\s+Integrity:(\d+\.\d+)\s+\|\s+Stealth:(\d+\.\d+)"
    )
    
    # Matches: 42959.4s 1125 [Layer 6 Semantic Audit] HALLUCINATION DETECTED
    hallucination_pattern = re.compile(r"(\d+\.?\d*)s\s+\d+\s+\[Layer 6 Semantic Audit\] HALLUCINATION DETECTED")

    print(f"Reading {file_path}...")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # Check for rewards
            reward_match = reward_pattern.search(line)
            if reward_match:
                ts, outcome, integrity, stealth = reward_match.groups()
                total = float(outcome) + float(integrity) + float(stealth)
                data.append({
                    "timestamp": float(ts),
                    "outcome": float(outcome),
                    "integrity": float(integrity),
                    "stealth": float(stealth),
                    "total": total
                })
                
            # Check for hallucinations
            hal_match = hallucination_pattern.search(line)
            if hal_match:
                hallucinations.append(float(hal_match.group(1)))

    return pd.DataFrame(data), hallucinations

def create_plots(df, hallucinations, output_name="training_performance.png"):
    """
    Generates a multi-panel plot showing reward progression and audit failures.
    """
    if df.empty:
        print("No reward data found to plot.")
        return

    # Use a modern style
    plt.style.use('dark_background')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    
    # Apply rolling average for smoothing
    window = max(5, len(df) // 20)
    df['total_smooth'] = df['total'].rolling(window=window).mean()
    df['outcome_smooth'] = df['outcome'].rolling(window=window).mean()
    df['integrity_smooth'] = df['integrity'].rolling(window=window).mean()
    df['stealth_smooth'] = df['stealth'].rolling(window=window).mean()

    # --- TOP PLOT: Total Reward & Components ---
    ax1.plot(df['timestamp'], df['total'], color='purple', alpha=0.2, label='Raw Total')
    ax1.plot(df['timestamp'], df['total_smooth'], color='#a855f7', linewidth=3, label='Total Reward (Smoothed)')
    
    ax1.plot(df['timestamp'], df['outcome_smooth'], color='#10b981', linestyle='--', alpha=0.8, label='Outcome (Targeting)')
    ax1.plot(df['timestamp'], df['integrity_smooth'], color='#38bdf8', linestyle=':', alpha=0.8, label='Integrity (XAI)')
    ax1.plot(df['timestamp'], df['stealth_smooth'], color='#f59e0b', linestyle='-.', alpha=0.8, label='Stealth (Efficiency)')

    ax1.set_title("SOC Duel: Multi-Agent Reward Progression", fontsize=16, pad=15)
    ax1.set_ylabel("Reward Value (Max 0.99)", fontsize=12)
    ax1.legend(loc='upper left', fontsize=9, ncol=2)
    ax1.grid(True, linestyle='--', alpha=0.3)
    ax1.set_ylim(0, 1.0)

    # --- BOTTOM PLOT: Hallucination Events ---
    if hallucinations:
        # Create a density plot/histogram of hallucinations over time
        ax2.hist(hallucinations, bins=50, color='#f43f5e', alpha=0.3, label='Hallucination Frequency')
        # Add a cumulative line
        counts, bin_edges = np.histogram(hallucinations, bins=100)
        cumulative = np.cumsum(counts)
        ax2_twin = ax2.twinx()
        ax2_twin.plot(bin_edges[1:], cumulative, color='#f43f5e', linewidth=2, label='Total Hallucinations (Cumulative)')
        ax2_twin.set_ylabel("Cumulative Audit Failures", color='#f43f5e')
        ax2_twin.tick_params(axis='y', labelcolor='#f43f5e')

    ax2.set_title("Layer 6 Semantic Audit: Hallucination Timeline", fontsize=14)
    ax2.set_xlabel("Inference Timestamp (Seconds)", fontsize=12)
    ax2.set_ylabel("Event Count", fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.3)
    
    # Text summary overlay
    stats_text = (
        f"Total Samples: {len(df)}\n"
        f"Max Reward: {df['total'].max():.3f}\n"
        f"Avg Reward: {df['total'].mean():.3f}\n"
        f"Total Audit Failures: {len(hallucinations)}"
    )
    fig.text(0.15, 0.02, stats_text, fontsize=10, bbox=dict(facecolor='black', alpha=0.5))

    plt.tight_layout()
    plt.savefig(output_name, dpi=150)
    print(f"Graph saved as {output_name}")
    plt.show()

if __name__ == "__main__":
    # Path to your uploaded file
    FILE_PATH = "Logs.txt"
    
    try:
        df, hallucinations = parse_log_file(FILE_PATH)
        create_plots(df, hallucinations)
    except FileNotFoundError:
        print(f"Error: {FILE_PATH} not found. Please ensure the file is in the same directory.")
    except Exception as e:
        print(f"An error occurred: {e}")

---

## 📉 Section 9: Training Loss Curve
### Plot Policy Loss Over Training Steps

Reads the `log_history` from the trainer state and plots the **GRPO policy loss** across all recorded steps. A filled area beneath the curve aids in visual interpretation of loss magnitude. The figure is saved as `soc_loss_curve.png`.

In [ ]:
# ==============================================================================
# CELL 10: Plot Training Loss
# ==============================================================================
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [log['step'] for log in log_history if 'loss' in log]
losses = [log['loss'] for log in log_history if 'loss' in log]

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, label='Policy Loss (GRPO)', color='#3b82f6', linewidth=2)
plt.fill_between(steps, losses, color='#3b82f6', alpha=0.1)

plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('SOC Model Loss Funaction')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.savefig('soc_loss_curve.png', dpi=120)
plt.show()

print(f"✅ Loss graph generated over {len(steps)} steps.")

---

## 🎯 Section 10: Epoch-Level Accuracy & Reward
### Aggregate Episode Log into Per-Epoch Accuracy & Mean Reward

Post-processes the `EPISODE_LOG` accumulated during training. Divides completions into 10 epoch bins and computes for each:
- **Operational Accuracy (%)**: Fraction of moves achieving a total reward > 0.5 (i.e., successfully executed without hallucination or poor targeting).
- **Mean Reward**: Average composite reward across all completions in the bin.

A dual-axis plot displays both metrics together — accuracy on the left y-axis (green) and mean reward on the right (purple). Saved as `soc_accuracy_epochs.png`.

In [ ]:
# ==============================================================================
# CELL 11: Plot Accuracy & Reward Progression (Aggregated by Epoch)
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np

# We define "Accuracy" as the percentage of moves that achieved a Reward > 0.5
# (Meaning they successfully navigated the topology without hallucinating)
total_completions = len(EPISODE_LOG)
steps_per_epoch = max(1, total_completions // 10) # Divide history into 10 epoch bins

epoch_accuracy = []
epoch_reward = []

for i in range(10):
    start = i * steps_per_epoch
    end = (i + 1) * steps_per_epoch
    batch = EPISODE_LOG[start:end]
    
    if batch:
        # Accuracy: % of completions with total reward > 0.5
        acc = len([e for e in batch if e['total'] > 0.5]) / len(batch)
        # Avg Reward
        avg_r = np.mean([e['total'] for e in batch])
        
        epoch_accuracy.append(acc * 100) # Convert to percentage
        epoch_reward.append(avg_r)

epochs_range = np.arange(1, len(epoch_accuracy) + 1)

fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot Accuracy (Success Rate)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Operational Accuracy (%)', color='#10b981')
ax1.plot(epochs_range, epoch_accuracy, color='#10b981', marker='o', linewidth=3, label='Success Rate %')
ax1.tick_params(axis='y', labelcolor='#10b981')
ax1.set_ylim(0, 100)

# Create a secondary axis for Mean Reward
ax2 = ax1.twinx()
ax2.set_ylabel('Mean Reward', color='#8b5cf6')
ax2.plot(epochs_range, epoch_reward, color='#8b5cf6', marker='s', linestyle='--', linewidth=2, label='Avg Reward')
ax2.tick_params(axis='y', labelcolor='#8b5cf6')
ax2.set_ylim(0, 1.0)

plt.title("SOC Intelligence: Accuracy & Reward per Epoch")
fig.tight_layout()
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.savefig('soc_accuracy_epochs.png', dpi=120)
plt.show()

print("✅ Saved soc_accuracy_epochs.png showing progress over 10 segments.")

---

## 💾 Section 11: Save Artifacts & Shutdown
### Persist LoRA Adapter & Gracefully Stop the Simulation Server

Saves the fine-tuned **LoRA adapter weights and tokenizer** to `/kaggle/working/soc_lora` using `save_pretrained`. These can be merged with the base model later for inference or uploaded to the HuggingFace Hub.

Then gracefully terminates the background simulation server subprocess (with a 5-second timeout before force-killing), freeing up the GPU and network port cleanly.

In [ ]:
# ==============================================================================
# CELL 12: Save Adapter + Tear Down Server
# ==============================================================================
LORA_PATH = '/kaggle/working/soc_lora'
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f'LoRA adapter saved to {LORA_PATH}')

if SERVER_PROC.poll() is None:
    SERVER_PROC.terminate()
    try:
        SERVER_PROC.wait(timeout=5)
    except subprocess.TimeoutExpired:
        SERVER_PROC.kill()
print('SOC Environment Server stopped.')


---

## ✅ Notebook Complete

The full SOC Duel GRPO pipeline has finished. You should now have:
- `soc_lora/` — fine-tuned LoRA adapter weights
- `soc_loss_curve.png` — policy loss progression
- `soc_accuracy_epochs.png` — epoch-level accuracy & reward chart
- `training_performance.png` — reward component dashboard (if logs were present)

> To use the adapter for inference, reload the base model with `FastLanguageModel.from_pretrained()` and merge the adapter via `model.merge_and_unload()`.